In [ ]:
!pip -q install -U transformers accelerate sentencepiece


In [ ]:
import re
import torch
from transformers import AutoProcessor, AutoModelForCausalLM


In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, switch to a GPU runtime before running this notebook.")

device = "cuda"
dtype = torch.float16

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto"
)
model.eval()

print(f"Loaded {MODEL_ID} on {device} with dtype={dtype}.")


In [ ]:
def _extract_from_parsed(parsed):
    thought = None
    answer = None

    if isinstance(parsed, dict):
        for key in ("thought", "thinking", "reasoning", "analysis"):
            if key in parsed and parsed[key] is not None:
                thought = parsed[key]
                break
        for key in ("response", "answer", "final", "final_answer", "text", "content"):
            if key in parsed and parsed[key] is not None:
                answer = parsed[key]
                break
    elif isinstance(parsed, (list, tuple)) and len(parsed) >= 2:
        thought, answer = parsed[0], parsed[1]

    if thought is not None:
        thought = str(thought).strip()
    if answer is not None:
        answer = str(answer).strip()

    return thought, answer


def split_gemma_response(raw_text):
    parsed = None
    thought = None
    answer = None

    if hasattr(processor, "parse_response"):
        try:
            parsed = processor.parse_response(raw_text)
            thought, answer = _extract_from_parsed(parsed)
        except Exception:
            parsed = None

    if thought is not None or answer is not None:
        return thought or "", answer or "", parsed

    fallback_patterns = [
        r"<\|channel\|>thought\n(?P<thought>.*?)(?:<\|/?channel\|>|<channel\|>)(?P<answer>.*)",
        r"<\|channel\|>thought\n(?P<thought>.*?)(?:<\|end\|>|<eos>|</think>)(?P<answer>.*)",
    ]

    for pattern in fallback_patterns:
        match = re.search(pattern, raw_text, flags=re.DOTALL)
        if match:
            thought = match.group("thought").strip()
            answer = match.group("answer").strip()
            return thought, answer, parsed

    return "", raw_text.strip(), parsed


In [ ]:
def render_visible(text):
    return text.encode("unicode_escape").decode("ascii")


def dump_token_table(token_ids, tokenizer, limit=None):
    rows = []
    ids = token_ids if limit is None else token_ids[:limit]
    for idx, token_id in enumerate(ids):
        token_text = tokenizer.decode([token_id], skip_special_tokens=False)
        rows.append((idx, int(token_id), render_visible(token_text)))
    return rows


In [ ]:
def _sample_next_token(logits, do_sample=False, temperature=1.0, top_p=0.95, top_k=64):
    if not do_sample:
        return torch.argmax(logits, dim=-1, keepdim=True)

    logits = logits / max(temperature, 1e-5)

    if top_k is not None and top_k > 0:
        top_k = min(top_k, logits.shape[-1])
        values, _ = torch.topk(logits, top_k)
        min_values = values[:, -1].unsqueeze(-1)
        logits = torch.where(logits < min_values, torch.full_like(logits, float("-inf")), logits)

    if top_p is not None and top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs > top_p
        sorted_mask[:, 1:] = sorted_mask[:, :-1].clone()
        sorted_mask[:, 0] = False
        scatter_mask = torch.zeros_like(logits, dtype=torch.bool)
        scatter_mask.scatter_(1, sorted_indices, sorted_mask)
        logits = torch.where(scatter_mask, torch.full_like(logits, float("-inf")), logits)

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


In [ ]:
@torch.inference_mode()
def ask_gemma(
    question,
    system_prompt="You are a helpful assistant.",
    do_sample=False,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    emergency_max_new_tokens=8192,
):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

    model_inputs = processor(text=prompt_text, return_tensors="pt")
    input_ids = model_inputs["input_ids"].to(model.device)
    attention_mask = model_inputs["attention_mask"].to(model.device)

    generated_ids = input_ids
    eos_token_id = processor.tokenizer.eos_token_id
    past_key_values = None

    for step in range(emergency_max_new_tokens):
        outputs = model(
            input_ids=generated_ids if past_key_values is None else generated_ids[:, -1:],
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )

        next_token = _sample_next_token(
            outputs.logits[:, -1, :],
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )

        generated_ids = torch.cat([generated_ids, next_token], dim=-1)
        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=-1)
        past_key_values = outputs.past_key_values

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

    full_text = processor.decode(generated_ids[0], skip_special_tokens=False)
    generated_token_ids = generated_ids[0][input_ids.shape[-1]:].tolist()
    raw_generation = processor.decode(generated_ids[0][input_ids.shape[-1]:], skip_special_tokens=False)
    reasoning, final_answer, parsed = split_gemma_response(raw_generation)
    token_rows = dump_token_table(generated_token_ids, processor.tokenizer)

    print("=" * 100)
    print("QUESTION")
    print(question)
    print()
    print("RAW GENERATION")
    print(raw_generation)
    print()
    print("RAW GENERATION (VISIBLE ESCAPES)")
    print(render_visible(raw_generation))
    print()
    print("GENERATED TOKENS")
    for idx, token_id, token_text in token_rows:
        print(f"{idx:04d}  {token_id:>8}  {token_text}")
    print()
    print("REASONING TRACE")
    print(reasoning if reasoning else "[No reasoning block could be extracted]")
    print()
    print("FINAL ANSWER")
    print(final_answer)
    print("=" * 100)

    return {
        "question": question,
        "prompt_text": prompt_text,
        "full_text": full_text,
        "generated_token_ids": generated_token_ids,
        "generated_tokens": token_rows,
        "raw_generation": raw_generation,
        "reasoning": reasoning,
        "final_answer": final_answer,
        "parsed": parsed,
    }


In [ ]:
result = ask_gemma("What is 5 + 5?")


In [ ]:
                                                    
                                          

question = "Explain why the sky appears blue during the day."

result = ask_gemma(
    question,
    do_sample=False,
)
